In [1]:
## Load packages

import pandas as pd
import numpy as np
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.model_selection import train_test_split
from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, hamming_loss
from transformers import EvalPrediction
from datasets import Dataset
import torch
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from itertools import product
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn

os.environ["TRANSFORMERS_CACHE"] = ""
os.environ["HF_HOME"]=""
device = 'cuda'
os.environ["TOKENIZERS_PARALLELISM"] = "false"

/home/skunk/routing2/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/home/skunk/routing2/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


# Data/Model Processing

In [2]:
## evalution metrics

## source: https://jesusleal.io/2021/04/21/Longformer-multilabel-classification/
def multi_label_metrics(predictions, labels, threshold=0.5):
    # first, apply sigmoid on predictions which are of shape (batch_size, num_labels)
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions))
    # next, use threshold to turn them into integer predictions
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= threshold)] = 1
    # finally, compute metrics
    y_true = np.zeros(labels.shape)
    y_true[np.where(labels >= threshold)] = 1 
    f1_micro_average = f1_score(y_true=y_true, y_pred=y_pred, average='micro')
    roc_auc = roc_auc_score(y_true, y_pred, average = 'micro')
    accuracy = accuracy_score(y_true, y_pred)
    # return as dictionary
    metrics = {'f1': f1_micro_average,
               'roc_auc': roc_auc,
               'accuracy': accuracy,
               'all_accuracy': (y_true == y_pred).mean()}
    return metrics

def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions, 
            tuple) else p.predictions
    result = multi_label_metrics(
        predictions=preds, 
        labels=p.label_ids)
    return result

In [3]:
seed = 4
base_model = "roberta-base"
params = {'seed': seed, 'model': base_model}
batch_size = 16
metric_name = "roc_auc"

In [4]:
# Load and process dataset
data = pd.read_pickle('dataset/routerbench_0shot.pkl')
data = data[[not s.lower().startswith('chinese') for s in data['eval_name'].values]] # remove the chinese questions

label_names = [str(s) for s in data.columns.values[3:14]]
id2label = {idx:label for idx, label in enumerate(label_names)}
label2id = {label:idx for idx, label in enumerate(label_names)}

inputs = [s[2:-2] for s in data['prompt']]
labels = (data.values[:,3:14]).astype('float')

cost_columns = [model+'|total_cost' for model in label_names]
#for c in data.columns:
    #if 'total_cost' in c:
        #cost_columns.append(c)
        
costs = np.array(data[cost_columns].values, dtype = 'float32')
means = costs.mean(axis = 0, keepdims = True)
stds = costs.std(axis = 0, keepdims = True)
costs_norm = (costs - means) / stds

train_idx, test_idx, train_inputs,\
    test_inputs, train_labels,\
    test_labels, train_costs, test_costs = train_test_split(np.arange(len(inputs)), \
                                   inputs, labels, costs_norm, train_size=0.8, random_state=seed)
                                   
train_idx, val_idx, train_inputs,\
    val_inputs, train_labels,\
    val_labels, train_costs, val_costs = train_test_split(np.arange(len(train_inputs)), \
                                   train_inputs, train_labels, train_costs, train_size=0.9, random_state=seed)

data_train = Dataset.from_dict({"input": train_inputs, "label": train_labels})
data_val = Dataset.from_dict({"input": val_inputs, "label": val_labels})
data_train_cost = Dataset.from_dict({"input": train_inputs, "label": train_costs})
data_val_cost = Dataset.from_dict({"input": val_inputs, "label": val_costs})
data_test = Dataset.from_dict({"input": test_inputs, "label": test_labels})
data_test_cost = Dataset.from_dict({"input": test_inputs, "label": test_costs})
data_all = Dataset.from_dict({"input": inputs, "label": labels})
data_all_cost = Dataset.from_dict({"input": inputs, "label": costs})

## Training meta model for correctness

In [5]:
# Loading base models

model_name = params["model"]
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, 
                                                               problem_type="multi_label_classification", 
                                                               num_labels=len(label_names),
                                                               id2label=id2label,
                                                               label2id=label2id)


# Tokenizing dataset
def preprocess_function(examples):
    return tokenizer(examples["input"], truncation=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_train = data_train.map(preprocess_function, batched=True)
tokenized_val = data_val.map(preprocess_function, batched=True)
tokenized_test = data_test.map(preprocess_function, batched=True)
tokenized_all = data_all.map(preprocess_function, batched=True)
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")
tokenized_all.set_format("torch")
tokenized_val.set_format("torch")

# Setting up training module
args = TrainingArguments(
    "models/roberta-perf",
    evaluation_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model=metric_name,
    seed = seed,
    report_to="none"
)

trainer =Trainer(
    model,
    args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.weight', 'classifier.out_proj.weight', 'classifier.out_proj.bias', 'classifier.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/25712 [00:00<?, ? examples/s]

Map:   0%|          | 0/2857 [00:00<?, ? examples/s]

Map:   0%|          | 0/7143 [00:00<?, ? examples/s]

Map:   0%|          | 0/35712 [00:00<?, ? examples/s]

In [7]:
# Training

trainer.train()

# the all_accuracy should be around 72%. 

You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,F1,Roc Auc,Accuracy,All Accuracy
1,0.582700,0.570902,0.754498,0.702889,0.100805,0.713017
2,0.563100,0.566579,0.760819,0.708222,0.096955,0.718968
3,0.552700,0.565710,0.761446,0.713108,0.097655,0.722468
4,0.530300,0.575154,0.758838,0.713757,0.113406,0.721927
5,0.507700,0.585896,0.756233,0.709975,0.107105,0.718427


TrainOutput(global_step=4020, training_loss=0.5458677894440457, metrics={'train_runtime': 2213.4807, 'train_samples_per_second': 58.08, 'train_steps_per_second': 1.816, 'total_flos': 2.5643897059550496e+16, 'train_loss': 0.5458677894440457, 'epoch': 5.0})

In [8]:
# Model wise accuracy

def acc_models(data_test, logits, models):
    labels = np.array(np.array(data_test['label']) >= 0.5, dtype = 'float')
    pred = (np.array(logits) >= 0).astype('float')
    
    for i, m in enumerate(models):
        
        acc = labels[:,i].mean()
        acc_meta = (labels[:, i] == pred[:, i]).mean()
        print(m.split('/')[-1], 'Accuracy', acc, 'Meta acc', acc_meta, 'baseline', max(acc, 1 - acc))

        
acc_models(data_test, trainer.predict(tokenized_test).predictions, label_names)

WizardLM-13B-V1.2 Accuracy 0.4605907881842363 Meta acc 0.6553268934621308 baseline 0.5394092118157636
claude-instant-v1 Accuracy 0.6514069718605627 Meta acc 0.6861262774744505 baseline 0.6514069718605627
claude-v1 Accuracy 0.6824863502729945 Meta acc 0.7086658266834663 baseline 0.6824863502729945
claude-v2 Accuracy 0.6936861262774745 Meta acc 0.7090858182836344 baseline 0.6936861262774745
gpt-3.5-turbo-1106 Accuracy 0.6745065098698027 Meta acc 0.6913061738765225 baseline 0.6745065098698027
gpt-4-1106-preview Accuracy 0.8548229035419291 Meta acc 0.8527229455410892 baseline 0.8548229035419291
code-llama-instruct-34b-chat Accuracy 0.21027579448411032 Meta acc 0.8210835783284335 baseline 0.7897242055158897
llama-2-70b-chat Accuracy 0.3614727705445891 Meta acc 0.7586448271034579 baseline 0.638527229455411
mistral-7b-chat Accuracy 0.30939381212375755 Meta acc 0.7090858182836344 baseline 0.6906061878762424
mixtral-8x7b-chat Accuracy 0.5678286434271315 Meta acc 0.6701665966680667 baseline 0.56

In [ ]:
# Saving the logits

result = trainer.predict(tokenized_all)
logits = result.predictions

data_path = '/dataset'

for i, m in enumerate(label_names):
    data[m+'_logits'] = logits[:, i]
data.to_pickle(os.path.join(data_path, 'roberta_base.pkl'))

In [10]:
data_path = 'dataset'
result = trainer.predict(tokenized_all)
logits = result.predictions
for i, m in enumerate(label_names):
    data[m+'_logits'] = logits[:, i]
data.to_pickle(os.path.join(data_path, 'roberta_base_acc_sms.pkl'))

## Training meta model for cost prediction

Might have to restart the kernel

In [12]:

## Loading base model
model_name = params["model"]
tokenizer = AutoTokenizer.from_pretrained(model_name)
def compute_metrics_for_regression(eval_pred: EvalPrediction):
    logits, labels = eval_pred
    # labels = labels.reshape(-1, 1) - PM taking this out as we have 1x10 label vector to compare
    
    #mse = mean_squared_error(labels, logits)
    mse = np.mean(np.square(labels-logits))
    # TODO MAE should be reported in a scaled back way possibly
    #mae = mean_absolute_error(labels, logits, multioutput="raw_values")
    #avg_mae = mean_absolute_error(labels, logits)
    #mae_by_model = [x*1000 for x in mae]
    #r2 = r2_score(labels, logits)
    
    return {"mse": mse}

#model = AutoModelForSequenceClassification.from_pretrained(model_name, 
                                                               #problem_type="multi_label_classification", 
                                                               #num_labels=len(label_names),
                                                               #id2label=id2label,
                                                               #label2id=label2id)
model = AutoModelForSequenceClassification.from_pretrained(base_model, 
                                                           problem_type="regression", 
                                                           num_labels=len(label_names),
                                                           )


# Tokenizing dataset
def preprocess_function(examples):
    return tokenizer(examples["input"], truncation=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_train = data_train_cost.map(preprocess_function, batched=True)
tokenized_val = data_val_cost.map(preprocess_function, batched=True)
tokenized_test = data_test_cost.map(preprocess_function, batched=True)
tokenized_all = data_all_cost.map(preprocess_function, batched=True)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.weight', 'classifier.out_proj.weight', 'classifier.out_proj.bias', 'classifier.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/25712 [00:00<?, ? examples/s]

Map:   0%|          | 0/2857 [00:00<?, ? examples/s]

Map:   0%|          | 0/7143 [00:00<?, ? examples/s]

Map:   0%|          | 0/35712 [00:00<?, ? examples/s]

In [13]:
# Setting up training module
args = TrainingArguments(
    "models/roberta-router",
    evaluation_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=5,
    metric_for_best_model = 'mse',
    weight_decay=0.01,
    load_best_model_at_end=True,
    seed = seed,
    report_to="none"
)


In [14]:
## MSE loss for cost prediction

#from torch.nn import MSELoss

#loss_fn = MSELoss()
#class CustomTrainer(Trainer):
    #def __init__(self, *args, **kwargs):
        #super().__init__(*args, **kwargs)

    #def compute_loss(self, model, inputs, return_outputs=False):
        #labels = inputs.pop("labels")
        #outputs = model(**inputs)
        
        # Calculate custom loss
        #loss = loss_fn(outputs.logits, labels)

        #return (loss, outputs) if return_outputs else loss


trainer =Trainer(
    model,
    args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics_for_regression,
    data_collator=data_collator
)

# Training
trainer.train()

You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Mse
1,0.205800,0.144953,0.144941
2,0.119600,0.137102,0.137095
3,0.120000,0.125037,0.125030
4,0.101700,0.122171,0.122164
5,0.097400,0.122624,0.122616


TrainOutput(global_step=4020, training_loss=0.12111718930415254, metrics={'train_runtime': 2219.7974, 'train_samples_per_second': 57.915, 'train_steps_per_second': 1.811, 'total_flos': 2.5643897059550496e+16, 'train_loss': 0.12111718930415254, 'epoch': 5.0})

In [11]:
print(len(labels))

35712


In [10]:
# Saving results

result = trainer.predict(tokenized_all)
costs_pred = result.predictions

costs_pred = costs_pred * stds + means # destandardize it

data_path = 'dataset'
data_save = pd.read_pickle(os.path.join(data_path, 'roberta_base_acc_sms.pkl'))
data_save.columns
for i, m in enumerate(label_names):
    data_save[m+'_cost-pred'] = costs_pred[:, i]
    
data_save.to_pickle(os.path.join(data_path, 'roberta_base_v3.pkl'))

In [12]:
data_save.to_pickle(os.path.join(data_path, 'roberta_base_v3.pkl'))

## Training ERM model

In [ ]:
## Setting up parameters
params = {'seed': seed, 'model': base_model}
batch_size = 16
metric_name = "f1"
#data_path = '/skunk-pod-storage-smaity-40umich-2eedu-pvc/minimax-routing_v2/routerbench/dataset'

## Loading and preprocessing dataset
#data = pd.read_pickle(os.path.join(data_path, 'routerbench_0shot.pkl'))
data = pd.read_pickle('dataset/routerbench_0shot.pkl')
data = data[[not s.lower().startswith('chinese') for s in data['eval_name'].values]]

label_names = [str(s) for s in data.columns.values[3:14]]
#id2label = {idx:label for idx, label in enumerate(label_names)}
#label2id = {label:idx for idx, label in enumerate(label_names)}

inputs = [s[2:-2] for s in data['prompt']]
labels = (data.values[:,3:14] > 0.5).astype('float')
costs = []
for m in label_names:
    costs.append(data[m + '|total_cost'].values)
costs = np.array(costs).T
ls = [0.282, 0.349, 0.544, 0.691, 0.772] # change the convex combination accordingly
l = 0.282/8
label_proxy = (1 - l) * labels - l * 1000 * costs  # combining the risks
label_proxy -= label_proxy.min() # making them positive
label_proxy = label_proxy / label_proxy.max() # normalizing to max value = 1

In [ ]:
# Train test split 

train_idx, test_idx, train_inputs,\
    test_inputs, train_labels,\
    test_labels = train_test_split(np.arange(len(inputs)), \
                                   inputs, label_proxy, train_size=0.8, random_state=seed)

data_train = Dataset.from_dict({"input": train_inputs, "label": train_labels})
data_test = Dataset.from_dict({"input": test_inputs, "label": test_labels})
data_all = Dataset.from_dict({"input": inputs, "label": label_proxy})


## Loading base model
model_name = params["model"]
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name,
                                                               num_labels=len(label_names))

# Tokenizing dataset
def preprocess_function(examples):
    return tokenizer(examples["input"], truncation=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_train = data_train.map(preprocess_function, batched=True)
tokenized_test = data_test.map(preprocess_function, batched=True)
tokenized_all = data_all.map(preprocess_function, batched=True)

In [ ]:
# Setting up training module
args = TrainingArguments(
    "models/roberta-router",
    evaluation_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    seed = seed,
)

trainer =Trainer(
    model,
    args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Training
trainer.train()

In [ ]:
# Logits for the combined risks

result = trainer.predict(tokenized_all)
logits = result.predictions

In [ ]:
data_path = 'dataset'
data_sms = pd.read_pickle('dataset/sms_roberta_base_v3.pkl')

for i, m in enumerate(label_names):
    data_sms[m+'_logits-l:' + str(l)] = logits[:, i]
    
data_sms.to_pickle('dataset/sms_roberta_base_v3.pkl')